[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch04-data-engineering/03-du-lieu-dung-va-nhat-quan.ipynb)

# Dữ liệu đúng và giữ được đúng: khế ước, rò rỉ, trôi dữ liệu và nợ dữ liệu

Chương 4 mở đầu bằng một mô hình chấm điểm tín dụng bỗng từ chối cả một vùng, trong khi không có sự
cố nào được báo. Đó là kiểu thất bại mà sổ tay này dựng lại nhiều lần: dữ liệu vượt qua mọi kiểm tra
cú pháp, mô hình vẫn trả về điểm số, và hành vi đã học âm thầm hỏng. Mỗi phần dưới đây tái tạo một
kiểu hỏng như vậy trên dữ liệu nhỏ, rồi dựng phép kiểm tra bắt được nó.

**Bạn sẽ làm:**
1. tái tạo thác lỗi dữ liệu của mã bưu chính 2139, rồi viết một khế ước dữ liệu (data contract) chặn
   nó;
2. đo xem rò rỉ qua lằn ranh giữa các tập làm độ chính xác trông tốt hơn thật bao nhiêu;
3. chạy bốn bậc báo động trên một luồng 50,000 bản ghi mỗi giây;
4. làm kiểm định K-S và chỉ số ổn định phân phối (population stability index, PSI) bằng tay, và xem chúng nhạy tới đâu;
5. phân biệt ba kiểu trôi phân phối bằng những gì mỗi phép theo dõi nhìn thấy;
6. giữ nhất quán giữa huấn luyện và phục vụ, và viết phép biến đổi chạy lại không đổi kết quả;
7. đo xem nhãn sai và giá trị thiếu làm mô hình kém đi bao nhiêu;
8. chấm một danh mục dữ liệu theo các ngưỡng nợ dữ liệu (data debt) của chương.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import hashlib
import re
from datetime import date

import pandas as pd
from scipy import stats
from sklearn.datasets import load_digits
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.neighbors import KNeighborsClassifier

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # thứ tự màu cố định
dat_hat_giong()
print("sẵn sàng")

## 1. Thác lỗi bắt đầu từ một mã bưu chính

Ví dụ 1.1 của chương: một nhóm phía trên đổi cột `zip_code` từ số nguyên sang chuỗi (`"02139"`) để
hỗ trợ mã quốc tế, nhưng không cập nhật khế ước dữ liệu phía dưới. Một phép biến đổi cũ vẫn ép chuỗi
về số nguyên trước khi tuần tự hoá (serialization), nên mất số 0 ở đầu (`2139`). Mô hình coi một mã
quen thuộc là một hạng mục chưa từng thấy, tức rủi ro cao.

Sổ tay dựng một mô hình đồ chơi: rủi ro của mỗi đơn là rủi ro trung bình của vùng, học từ dữ liệu
lịch sử, và một mã chưa từng thấy nhận mức rủi ro thận trọng 0.5. Đơn được duyệt khi rủi ro dưới 0.2.
Các con số rủi ro là **giả định**.

In [ ]:
rng = np.random.default_rng(1)
RUI_RO_VUNG = {"02139": 0.04, "10001": 0.07, "94103": 0.06, "60601": 0.09, "30301": 0.08}  # giả định
RUI_RO_LA = 0.5                                # mã chưa từng thấy: giả định thận trọng
don = pd.DataFrame({"ma_buu_chinh": rng.choice(list(RUI_RO_VUNG), 2000)})


def cham_diem(ma):
    return RUI_RO_VUNG.get(ma, RUI_RO_LA)


def ty_le_duyet(bang):
    duyet = bang["ma_buu_chinh"].map(cham_diem) < 0.2
    return duyet.groupby(bang["vung"]).mean().mul(100).round(1)


truoc = don.assign(vung=don["ma_buu_chinh"])
# phía trên đã đổi sang chuỗi; phép biến đổi cũ phía dưới vẫn làm int() rồi str()
sau = don.assign(vung=don["ma_buu_chinh"], ma_buu_chinh=don["ma_buu_chinh"].map(lambda m: str(int(m))))
print(pd.DataFrame({"% được duyệt, trước": ty_le_duyet(truoc), "% được duyệt, sau": ty_le_duyet(sau)}))
assert ty_le_duyet(sau)["02139"] == 0 and ty_le_duyet(truoc)["02139"] == 100

Không có ngoại lệ nào, không dòng log nào báo lỗi: `"2139"` là một chuỗi hợp lệ. Chỉ một vùng hỏng,
đúng vùng có số 0 ở đầu. Chương gọi đây là thác lỗi dữ liệu (data cascade): lỗi ở phía trên lan qua
đặc trưng, huấn luyện và triển khai, và chi phí sửa tăng theo số sản phẩm phía dưới đã dùng dữ liệu
hỏng.

Cách chặn là đặt một khế ước ở ranh giới: một bản mô tả bằng mã những gì mỗi cột được phép chứa,
chạy trước khi dữ liệu đi tiếp. Kiểm tra giản đồ (schema validation) kiểu này là một dạng của chất
lượng dữ liệu dưới dạng mã (data quality as code). Ví dụ của chương về khoảng hợp lệ của tuổi là 18–95, và một quy trình có
kiểm tra sẽ chặn triển khai khi thấy tuổi bằng 150.

In [ ]:
TUOI_MIN = sach(18, trich="valid range of 18–95, while")
TUOI_MAX = sach(95, trich="valid range of 18–95, while")
TUOI_LA = sach(150, trich="age values of 150, triggering")

KHE_UOC = {
    "ma_khach": {"kieu": str, "khong_rong": True, "duy_nhat": True},
    "ma_buu_chinh": {"kieu": str, "mau": r"\d{5}"},
    "tuoi": {"kieu": int, "khoang": (TUOI_MIN, TUOI_MAX)},
}


def kiem_tra(bang, khe_uoc):
    """Trả về danh sách vi phạm (cột, luật, số dòng). Danh sách rỗng nghĩa là dữ liệu qua được."""
    loi = []
    for cot, luat in khe_uoc.items():
        if cot not in bang:
            loi.append((cot, "thiếu cột", len(bang)))
            continue
        v = bang[cot]
        sai_kieu = (~v.map(lambda x: isinstance(x, luat["kieu"]))).sum()
        if sai_kieu:
            loi.append((cot, f"không phải {luat['kieu'].__name__}", int(sai_kieu)))
        if luat.get("khong_rong") and v.isna().sum():
            loi.append((cot, "rỗng", int(v.isna().sum())))
        if luat.get("duy_nhat") and v.duplicated().sum():
            loi.append((cot, "trùng", int(v.duplicated().sum())))
        if "mau" in luat:
            lech = (~v.astype(str).str.fullmatch(luat["mau"])).sum()
            if lech:
                loi.append((cot, f"không khớp mẫu {luat['mau']}", int(lech)))
        if "khoang" in luat:
            a, b = luat["khoang"]
            ngoai = (~pd.to_numeric(v, errors="coerce").between(a, b)).sum()
            if ngoai:
                loi.append((cot, f"ngoài khoảng {a}–{b}", int(ngoai)))
    return loi


def tao_ban_ghi(n, rng):
    return pd.DataFrame({"ma_khach": [f"k{i:05d}" for i in range(n)],
                         "ma_buu_chinh": rng.choice(list(RUI_RO_VUNG), n),
                         "tuoi": rng.integers(TUOI_MIN, TUOI_MAX + 1, n).astype(object)})


tot = tao_ban_ghi(1000, np.random.default_rng(2))
assert kiem_tra(tot, KHE_UOC) == []
hong = tot.copy()
hong["ma_buu_chinh"] = hong["ma_buu_chinh"].map(lambda m: str(int(m)))   # phép biến đổi cũ
hong.loc[7, "tuoi"] = TUOI_LA
hong.loc[8, "ma_khach"] = hong.loc[9, "ma_khach"]
for vi_pham in kiem_tra(hong, KHE_UOC):
    print("chặn:", vi_pham)

Khế ước bắt được cả ba lỗi cơ học: mã bưu chính mất số 0, tuổi 150, và một mã khách trùng. Nhưng nó
chỉ thấy những gì nó được dặn. Phần 4 cho thấy một tập dữ liệu hoàn hảo về mặt cơ học vẫn có thể hỏng
về ngữ nghĩa.

## 2. Lằn ranh tin cậy giữa các tập

Chương nói tập huấn luyện được phép định hình tham số, tập kiểm định được phép định hình các lựa chọn,
còn tập kiểm tra được giữ lại để ước lượng khả năng tổng quát hoá. Rò rỉ (leakage) xảy ra khi thông
tin vượt lằn ranh đó, chẳng hạn khi các biến thể tăng cường của cùng một bản ghi gốc rơi vào cả hai
phía.

Sổ tay lấy 1,797 ảnh chữ số 8×8 của `load_digits`, tạo cho mỗi ảnh gốc 4 biến thể tăng cường (đổi độ
sáng trong khoảng ±10 % và thêm nhiễu nhỏ), tức 5 bản ghi mỗi ảnh gốc. Rồi chia 80/20 theo hai cách: chia ngẫu nhiên trên từng bản ghi, và chia theo
nhóm, sao cho mọi biến thể của một ảnh gốc nằm cùng một phía.

**Dự đoán:** cách chia nào báo tỉ lệ lỗi thấp hơn, và con số nào gần với tỉ lệ lỗi trên một ảnh chưa
từng thấy?

In [ ]:
X_so, y_so = load_digits(return_X_y=True)
rng = np.random.default_rng(2)


def tang_cuong(X):
    """Độ sáng ±10 % và nhiễu Gauss nhỏ, trên thang điểm ảnh 0–16 của load_digits."""
    return np.clip(X * rng.uniform(0.9, 1.1, (len(X), 1)) + rng.normal(0, 0.5, X.shape), 0, 16)


bien_the = [X_so] + [tang_cuong(X_so) for _ in range(4)]
X_tc = np.concatenate(bien_the)
y_tc = np.tile(y_so, len(bien_the))
goc = np.tile(np.arange(len(X_so)), len(bien_the))     # bản ghi gốc của mỗi dòng
print(f"{len(X_tc):,} bản ghi từ {len(X_so):,} ảnh gốc")

cach_chia = {
    "chia ngẫu nhiên từng bản ghi": train_test_split(np.arange(len(X_tc)), test_size=0.2, random_state=0),
    "chia theo ảnh gốc": next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0).split(X_tc, y_tc, goc)),
}
LOI = {}
for ten, (hl, kt) in cach_chia.items():
    chung = len(np.intersect1d(goc[hl], goc[kt]))
    co_anh_em = np.isin(goc[kt], goc[hl]).mean()
    mo_hinh = KNeighborsClassifier(n_neighbors=1).fit(X_tc[hl], y_tc[hl])
    LOI[ten] = 100 * (1 - mo_hinh.score(X_tc[kt], y_tc[kt]))
    print(f"{ten}: {chung:,} ảnh gốc có mặt ở cả hai phía; {100 * co_anh_em:.0f} % bản ghi kiểm tra có biến thể "
          f"anh em trong tập huấn luyện; tỉ lệ lỗi trên tập kiểm tra {LOI[ten]:.2f} %")
    if ten == "chia theo ảnh gốc":
        assert chung == 0
    else:
        assert chung > 0
assert LOI["chia ngẫu nhiên từng bản ghi"] < LOI["chia theo ảnh gốc"]

Mô hình láng giềng gần nhất ghi nhớ: khi biến thể anh em của một ảnh kiểm tra nằm trong tập huấn
luyện, nó chỉ cần tìm lại gần như chính ảnh đó. Con số thấp hơn đo khả năng ghi nhớ, không
đo khả năng tổng quát hoá. Với case phát hiện từ khoá, chương nói chính vì thế mà chia theo người nói
và theo thời gian không phải chuyện sổ sách: đó là khác biệt giữa đo trí nhớ về những giọng quen và đo
hiệu năng trên người dùng thật.

Kiểu rò rỉ thứ hai là theo thời gian. Ví dụ của chương: với người dùng rời bỏ dịch vụ ngày 15 tháng 1,
mô hình phải dùng đặc trưng tính ngày 14 tháng 1, không phải giá trị hiện tại vốn đã phản ánh việc họ
rời đi. Sổ tay dựng 4,000 người dùng **giả định**, một nửa rời đi trong tháng, và đặc trưng "số ngày
kể từ lần hoạt động cuối", tính theo hai cách.

In [ ]:
rng = np.random.default_rng(3)
N_ND = 4000
roi_di = rng.random(N_ND) < 0.5
hoat_dong = rng.poisson(np.where(roi_di, 3.0, 5.0))               # số lần đăng nhập tuần trước
cuoi_dung_han = rng.exponential(np.where(roi_di, 4.0, 2.5))        # tính tới ngày trước khi dự đoán
cuoi_hien_tai = np.where(roi_di, cuoi_dung_han + rng.uniform(10, 30, N_ND), cuoi_dung_han)  # tính "hôm nay"
nhan = roi_di.astype(int)
hl, kt = train_test_split(np.arange(N_ND), test_size=0.3, random_state=0)
for ten, dac_trung in (("tính tại thời điểm dự đoán", cuoi_dung_han), ("tính bằng giá trị hôm nay", cuoi_hien_tai)):
    X = np.column_stack([hoat_dong, dac_trung])
    lr = LogisticRegression().fit(X[hl], nhan[hl])
    tren_kiem_tra = lr.score(X[kt], nhan[kt])
    # khi phục vụ, đặc trưng chỉ có thể là giá trị tại thời điểm dự đoán
    khi_phuc_vu = lr.score(np.column_stack([hoat_dong, cuoi_dung_han])[kt], nhan[kt])
    print(f"{ten:>26}: độ chính xác trên tập kiểm tra {100 * tren_kiem_tra:.1f} %, khi phục vụ thật {100 * khi_phuc_vu:.1f} %")

Đặc trưng tính bằng giá trị hôm nay gần như chứa sẵn nhãn: ai đã rời đi thì lâu rồi không hoạt động.
Tập kiểm tra cũng được tính theo cách đó nên nó không thấy gì bất thường. Chỉ khi phục vụ, lúc đặc
trưng buộc phải là giá trị tại thời điểm dự đoán, mô hình mới lộ ra nó đã học một quy tắc không tồn
tại. Giữ cho đặc trưng đúng theo mốc thời gian (point-in-time correctness) là một việc chính của kho
đặc trưng (feature store).

## 3. Bốn bậc báo động trên một luồng

Chương xếp cảnh báo theo mức nghiêm trọng. Nghiêm trọng nhất là quy trình ngừng hẳn: thông lượng
(throughput) bằng 0 quá 5 phút. Suy giảm là khi thông lượng tụt còn 80 % mức nền, tỉ lệ lỗi vượt 5 %,
hoặc chỉ số chất lượng lệch quá 2 độ lệch chuẩn so với dữ liệu huấn luyện. Ví dụ của chương: một hệ
thống gợi ý xử lý 50,000 bản ghi mỗi giây báo động khi thông lượng xuống dưới 40,000 quá 10 phút.

In [ ]:
NEN = sach(50_000, trich="50,000 records per second")
NGUONG = sach(40_000, trich="drops below 40,000 records per second for more than 10 minutes")
PHUT_SUY_GIAM = sach(10, trich="drops below 40,000 records per second for more than 10 minutes")
PHUT_NGUNG = sach(5, nguon="50.000 bản ghi/giây → báo động dưới 40.000 · 5 phút · 10 phút")
theo_sach("ngưỡng so với mức nền (%)", 100 * NGUONG / NEN, sach=80, trich="80 percent of baseline")

Sổ tay mô phỏng 6 giờ thông lượng theo từng phút: dao động trong ngày, nhiễu, và ba sự cố **giả
định**. Một lần tụt ngắn 6 phút, một lần tụt kéo dài 25 phút, và một lần ngừng hẳn 8 phút. Luật báo
động đòi điều kiện kéo dài liên tục đủ lâu, chứ không báo ngay ở phút đầu tiên.

**Dự đoán:** sự cố nào làm luật báo, và báo ở phút thứ mấy của sự cố?

In [ ]:
rng = np.random.default_rng(4)
phut = np.arange(360)
luu_luong = NEN * (1 + 0.08 * np.sin(2 * np.pi * phut / 240)) + rng.normal(0, 1200, len(phut))
SU_CO = {"tụt ngắn": (60, 66, 37_000), "tụt kéo dài": (150, 175, 36_000), "ngừng hẳn": (280, 288, 0)}  # giả định
for dau, cuoi, muc in SU_CO.values():
    luu_luong[dau:cuoi] = muc + (rng.normal(0, 800, cuoi - dau) if muc else 0)


def bao_dong(chuoi, dieu_kien, so_phut):
    """Phút đầu tiên mà điều kiện đã đúng liên tục quá `so_phut` phút."""
    lien_tiep, bao = 0, []
    for t, x in enumerate(chuoi):
        lien_tiep = lien_tiep + 1 if dieu_kien(x) else 0
        if lien_tiep == so_phut + 1:
            bao.append(t)
    return bao


bao_suy_giam = bao_dong(luu_luong, lambda x: 0 < x < NGUONG, PHUT_SUY_GIAM)
bao_ngung = bao_dong(luu_luong, lambda x: x == 0, PHUT_NGUNG)
for ten, (dau, cuoi, _) in SU_CO.items():
    bat = [t for t in bao_suy_giam + bao_ngung if dau <= t < cuoi]
    print(f"{ten} (phút {dau}–{cuoi - 1}): " + (f"báo ở phút {bat[0] - dau + 1} của sự cố" if bat else "không báo"))
assert not [t for t in bao_suy_giam if 60 <= t < 66] and bao_ngung == [280 + PHUT_NGUNG]

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(phut, luu_luong / 1000, color=MAU[0], lw=1.2)
ax.axhline(NEN / 1000, color="0.5", lw=0.8, ls=":")
ax.axhline(NGUONG / 1000, color=MAU[1], lw=1, ls="--")
ax.text(362, NEN / 1000, "mức nền", fontsize=8, va="center")
ax.text(362, NGUONG / 1000, "ngưỡng\n80 %", fontsize=8, va="center", color=MAU[1])
for t, nhan_bao in [(t, "suy giảm") for t in bao_suy_giam] + [(t, "ngừng") for t in bao_ngung]:
    ax.axvline(t, color="0.15", lw=1)
    ax.text(t + 6, 18, f"← báo: {nhan_bao}", fontsize=8, va="center")
for ten, (dau, cuoi, _) in SU_CO.items():
    ax.axvspan(dau, cuoi, color=MAU[3], alpha=0.2, lw=0)
    ax.text((dau + cuoi) / 2, 62, ten, fontsize=8, ha="center", zorder=5, bbox=dict(fc="white", ec="none", pad=1.5))
ax.set_xlim(0, 360)
ax.set_ylim(-2, 68)
ax.set_xlabel("phút")
ax.set_ylabel("nghìn bản ghi mỗi giây")
ax.set_title("Luật báo động cần điều kiện kéo dài (mô phỏng, giả định)")
plt.tight_layout()
plt.show()

Lần tụt ngắn không làm luật báo: chờ quá 10 phút là cách chương chừa chỗ cho dao động bình thường của
lưu lượng. Cái giá là sự cố kéo dài phải chạy hơn 10 phút rồi mới bị phát hiện.

Mỗi đặc trưng còn có hồ sơ chất lượng riêng. Ví dụ của chương: nếu `user_age` rỗng ở hơn 5 % số bản
ghi trong khi dữ liệu huấn luyện có chưa tới 1 % giá trị rỗng, gần như chắc chắn nguồn phía trên đã
hỏng.

In [ ]:
RONG_HL = sach(1, trich="less than 1 percent nulls")
RONG_BAO = sach(5, trich="more than 5 percent of records")
rng = np.random.default_rng(5)
ty_le_rong = np.clip(rng.normal(0.7, 0.15, 24), 0, None)          # % rỗng theo từng giờ: giả định
ty_le_rong[17] = 7.4                                               # giờ 17: nguồn phía trên hỏng
for gio in np.flatnonzero(ty_le_rong > RONG_BAO):
    print(f"giờ {gio}: user_age rỗng {ty_le_rong[gio]:.1f} %, vượt {RONG_BAO} % (huấn luyện: dưới {RONG_HL} %) → báo")
assert list(np.flatnonzero(ty_le_rong > RONG_BAO)) == [17]

## 4. Cơ học và ngữ nghĩa: kiểm định K-S và PSI

Chương phân biệt kiểm tra cơ học ("tuổi có phải số nguyên không?") với kiểm tra ngữ nghĩa (semantic
validation: "phân phối
tuổi có đang dịch chuyển không?"). Một tập dữ liệu có thể hoàn hảo về mặt cơ học mà hỏng về ngữ
nghĩa, chẳng hạn mọi người dùng bỗng 25 tuổi vì một giá trị mặc định bị đổi.

Napkin math 1.4 dùng kiểm định Kolmogorov-Smirnov: thống kê $\mathcal{D}_\text{KS}$ là khoảng cách lớn
nhất giữa hai hàm phân phối tích luỹ thực nghiệm, so với ngưỡng tới hạn ở mức ý nghĩa 0.05:

$$\mathcal{D}_\text{crit} \approx 1.36\sqrt{\frac{n_0 + n_t}{n_0\,n_t}}.$$

In [ ]:
HE_SO_KS = sach(1.36, nguon="1,36 · 0,061 · 0,08 · n = 1000")
N0 = NT = sach(1000, nguon="1,36 · 0,061 · 0,08 · n = 1000")
d_crit = HE_SO_KS * math.sqrt((N0 + NT) / (N0 * NT))
theo_sach("ngưỡng tới hạn D_crit", d_crit, sach=0.061, nguon="1,36 · 0,061 · 0,08 · n = 1000")
D_QUAN_SAT = sach(0.08, nguon="1,36 · 0,061 · 0,08 · n = 1000")
assert D_QUAN_SAT > d_crit
print(f"D_KS = {D_QUAN_SAT} > {d_crit:.4f}: bác bỏ giả thuyết không trôi, báo trôi")


def ks(a, b):
    """D_KS viết tay: đánh giá cả hai hàm phân phối tích luỹ tại mọi điểm dữ liệu."""
    a, b = np.sort(a), np.sort(b)
    x = np.concatenate([a, b])
    return np.max(np.abs(np.searchsorted(a, x, side="right") / len(a) - np.searchsorted(b, x, side="right") / len(b)))


rng = np.random.default_rng(6)
mau_a, mau_b = rng.lognormal(4, 0.8, 1000), rng.lognormal(4.1, 0.8, 1000)
assert np.isclose(ks(mau_a, mau_b), stats.ks_2samp(mau_a, mau_b).statistic)
print(f"D_KS viết tay {ks(mau_a, mau_b):.4f} = scipy {stats.ks_2samp(mau_a, mau_b).statistic:.4f}")

Ngưỡng tới hạn co lại theo $1/\sqrt{n}$. Vậy với cùng một độ dịch chuyển, kiểm định nhạy tới đâu khi
cỡ mẫu tăng? Sổ tay giữ phân phối nền của `session_duration` là log-chuẩn (một **giả định**), dịch
trung bình của logarit đi một lượng $\delta$, và lặp 200 lần ở mỗi cấu hình để đếm tỉ lệ lần kiểm định
báo trôi.

**Dự đoán:** với $\delta = 0$, tức không có trôi nào, kiểm định vẫn báo bao nhiêu phần trăm số lần?

In [ ]:
CAC_DELTA = [0.0, 0.01, 0.02, 0.05, 0.1, 0.2]
CAC_N = [100, 1000, 10_000]
rng = np.random.default_rng(7)
bao_ty_le = {}
for n in CAC_N:
    nguong = HE_SO_KS * math.sqrt(2 / n)
    bao_ty_le[n] = [np.mean([ks(rng.lognormal(4, 0.8, n), rng.lognormal(4 + d, 0.8, n)) > nguong for _ in range(200)])
                    for d in CAC_DELTA]
    print(f"n = {n:>6,}: tỉ lệ báo trôi " + ", ".join(f"δ={d}: {100 * p:.0f} %" for d, p in zip(CAC_DELTA, bao_ty_le[n])))

fig, ax = plt.subplots(figsize=(8, 3.8))
for n, mau in zip(CAC_N, MAU):
    ax.plot(CAC_DELTA, 100 * np.array(bao_ty_le[n]), "o-", color=mau, lw=2, ms=6, label=f"n₀ = nₜ = {n:,}")
ax.axhline(5, color="0.5", lw=0.8, ls=":")
ax.text(0.15, 6.5, "α = 5 %", fontsize=8, color="0.3")
ax.set_xlabel("độ dịch δ của trung bình log(session_duration)")
ax.set_ylabel("% số lần kiểm định báo trôi")
ax.set_title("Kiểm định K-S: cỡ mẫu càng lớn, dịch chuyển càng nhỏ cũng bị bắt (mô phỏng)")
ax.legend(loc="center right", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Khi không có trôi, kiểm định vẫn báo khoảng 5 % số lần: đó chính là nghĩa của mức ý nghĩa 0.05, và
trên một bảng theo dõi chạy mỗi giờ cho hàng trăm đặc trưng, nó là hàng chục báo động giả mỗi ngày. Khi
cỡ mẫu lớn, một dịch chuyển nhỏ tới mức không ảnh hưởng gì tới mô hình cũng bị báo. Vì vậy chương nói
một phép kiểm định thống kê nên khởi động một cuộc điều tra, và các ngưỡng phải được hiệu chỉnh theo
mức nền và theo bằng chứng về kết quả, không có ngưỡng chung cho mọi nơi.

PSI làm việc trên các ô (bin) của phân phối nền. Với $e_i$ và $a_i$ là tỉ lệ của ô $i$ trong dữ liệu nền
và dữ liệu hiện tại,

$$\text{PSI} = \sum_i (a_i - e_i)\ln\frac{a_i}{e_i} = D_\text{KL}(a \,\Vert\, e) + D_\text{KL}(e \,\Vert\, a),$$

tức PSI là phân kỳ KL (KL divergence) tính theo cả hai chiều. Bảng 10 của chương dùng hai mốc cho nợ độ
mới: PSI trên 0.1 là cảnh báo, trên 0.25 là nghiêm trọng.

In [ ]:
PSI_CANH_BAO = sach(0.1, trich="PSI vs. training baseline | > 0.1 | > 0.25")
PSI_NGHIEM_TRONG = sach(0.25, trich="PSI vs. training baseline | > 0.1 | > 0.25")


def ty_le_o(x, bien):
    dem = np.histogram(np.clip(x, bien[0], bien[-1]), bins=bien)[0]
    return np.maximum(dem / dem.sum(), 1e-6)


def psi(nen, hien_tai, so_o=10):
    bien = np.quantile(nen, np.linspace(0, 1, so_o + 1))
    e, a = ty_le_o(nen, bien), ty_le_o(hien_tai, bien)
    return float(np.sum((a - e) * np.log(a / e)))


def kl(p, q):
    return float(np.sum(p * np.log(p / q)))


rng = np.random.default_rng(8)
nen = rng.lognormal(4, 0.8, 20_000)
bien = np.quantile(nen, np.linspace(0, 1, 11))
moi = rng.lognormal(4.2, 0.8, 20_000)
e, a = ty_le_o(nen, bien), ty_le_o(moi, bien)
assert np.isclose(psi(nen, moi), kl(a, e) + kl(e, a))
for d in CAC_DELTA + [0.3, 0.5]:
    gia_tri = psi(nen, rng.lognormal(4 + d, 0.8, 20_000))
    muc = "nghiêm trọng" if gia_tri > PSI_NGHIEM_TRONG else "cảnh báo" if gia_tri > PSI_CANH_BAO else "ổn"
    print(f"δ = {d:<4}: PSI = {gia_tri:.4f} → {muc}")

Ngược với K-S, PSI gần như không phụ thuộc cỡ mẫu khi mẫu đã đủ lớn: nó đo độ lớn của dịch chuyển, chứ
không đo độ chắc chắn rằng có dịch chuyển. Một dịch chuyển mà K-S ở n = 10,000 gần như lần nào cũng báo
vẫn có thể nằm dưới mốc cảnh báo của PSI. Hai phép đo trả lời hai câu hỏi khác nhau.

Giờ đến trường hợp "mọi người dùng bỗng 25 tuổi". Một bản cập nhật đổi giá trị mặc định của biểu mẫu
đăng ký, nên 30 % người dùng mới (một **giả định**) được ghi tuổi 25. Mọi giá trị vẫn là số nguyên,
vẫn trong khoảng 18–95, không có giá trị rỗng.

**Dự đoán:** khế ước ở phần 1 có bắt được lỗi này không? K-S và PSI thì sao?

In [ ]:
rng = np.random.default_rng(9)
tuoi_nen = np.clip(np.round(TUOI_MIN + rng.gamma(4, 5, 20_000)), TUOI_MIN, TUOI_MAX).astype(int)
tuoi_moi = np.clip(np.round(TUOI_MIN + rng.gamma(4, 5, 20_000)), TUOI_MIN, TUOI_MAX).astype(int)
tuoi_moi[rng.random(20_000) < 0.3] = 25
ban_ghi_moi = tao_ban_ghi(20_000, np.random.default_rng(10)).assign(tuoi=tuoi_moi.astype(object))
ban_ghi_moi["tuoi"] = ban_ghi_moi["tuoi"].map(int).astype(object)
print("khế ước:", kiem_tra(ban_ghi_moi, KHE_UOC) or "qua, không vi phạm nào")
d_tuoi = ks(tuoi_nen, tuoi_moi)
print(f"K-S: D = {d_tuoi:.3f}, ngưỡng ở n = 20,000 là {HE_SO_KS * math.sqrt(2 / 20_000):.3f} → báo trôi")
print(f"PSI = {psi(tuoi_nen, tuoi_moi):.3f}, mốc nghiêm trọng của bảng 10 là {PSI_NGHIEM_TRONG}")

fig, ax = plt.subplots(figsize=(8, 3.4))
o = np.arange(TUOI_MIN, TUOI_MAX + 2) - 0.5
ax.hist(tuoi_nen, bins=o, color=MAU[0], alpha=0.55, label="dữ liệu nền")
ax.hist(tuoi_moi, bins=o, histtype="step", color=MAU[1], lw=1.5, label="sau bản cập nhật")
ax.annotate("30 % ghi tuổi 25", (25, np.sum(tuoi_moi == 25)), xytext=(45, np.sum(tuoi_moi == 25) * 0.85),
            fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.7))
ax.set_xlim(15, 85)
ax.set_xlabel("tuổi")
ax.set_ylabel("số người dùng")
ax.set_title("Hoàn hảo về cơ học, hỏng về ngữ nghĩa (mô phỏng, giả định)")
ax.legend(loc="upper right", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Khế ước để lọt vì nó kiểm tra từng bản ghi, mà từng bản ghi đều hợp lệ. Chỉ một phép kiểm tra trên cả
phân phối mới thấy được. Chương ghi một hệ thống bền phải kiểm tra cả vỏ chứa (cơ học) lẫn nội dung
(ngữ nghĩa).

## 5. Ba kiểu trôi phân phối, và phép theo dõi nào nhìn thấy chúng

Chương phân biệt ba kiểu trôi dữ liệu (data drift):

* **dịch chuyển hiệp biến (covariate shift):** $p(x)$ đổi, $p(y \mid x)$ giữ nguyên, như camera mới ở
  một bệnh viện;
* **dịch chuyển nhãn (label shift):** $p(y)$ đổi, $p(x \mid y)$ giữ nguyên, như tỉ lệ mắc bệnh đổi theo
  mùa;
* **trôi khái niệm (concept drift):** chính $p(y \mid x)$ đổi, như kẻ gian lận đổi cách tấn công.

Sổ tay huấn luyện một hồi quy logistic trên dữ liệu nền hai chiều, rồi đưa cho nó bốn tập phục vụ: một
tập mới từ đúng phân phối nền, và ba tập mang ba kiểu trôi. Mỗi tập được nhìn qua ba phép theo dõi: PSI
của một đặc trưng đầu vào, PSI của điểm số mô hình, và độ chính xác. Hai phép đầu không cần nhãn, phép
cuối cần. Mọi phân phối là **giả định**.

**Dự đoán:** kiểu trôi nào không phép theo dõi không cần nhãn nào nhìn thấy?

In [ ]:
rng = np.random.default_rng(11)
TAM0, TAM1, PI0 = np.array([0.0, 0.0]), np.array([2.0, 1.0]), 0.3


def hau_nghiem_nen(x):
    """p(y = 1 | x) của phân phối nền: hai cụm Gauss cùng hiệp phương sai đơn vị."""
    lo = np.log(PI0 / (1 - PI0)) + x @ (TAM1 - TAM0) - 0.5 * (TAM1 @ TAM1 - TAM0 @ TAM0)
    return 1 / (1 + np.exp(-lo))


def sinh_nen(n, pi=PI0):
    y = (rng.random(n) < pi).astype(int)
    return np.where(y[:, None] == 1, TAM1, TAM0) + rng.normal(size=(n, 2)), y


X_nen, y_nen = sinh_nen(20_000)
mo_hinh = LogisticRegression().fit(X_nen, y_nen)
diem_nen = mo_hinh.predict_proba(X_nen)[:, 1]

kich_ban = {}
kich_ban["không trôi"] = sinh_nen(20_000)
X_hb = rng.normal(size=(20_000, 2)) * 1.2 + np.array([1.5, -0.5])         # p(x) đổi
kich_ban["dịch chuyển hiệp biến"] = (X_hb, (rng.random(20_000) < hau_nghiem_nen(X_hb)).astype(int))
kich_ban["dịch chuyển nhãn"] = sinh_nen(20_000, pi=0.6)                   # p(y) đổi
X_kn, _ = sinh_nen(20_000)
goc_xoay = np.deg2rad(70)                                                 # p(y|x) đổi
xoay = np.array([[np.cos(goc_xoay), -np.sin(goc_xoay)], [np.sin(goc_xoay), np.cos(goc_xoay)]])
kich_ban["trôi khái niệm"] = (X_kn, (rng.random(20_000) < hau_nghiem_nen(X_kn @ xoay.T)).astype(int))

dong = []
for ten, (X, y) in kich_ban.items():
    diem = mo_hinh.predict_proba(X)[:, 1]
    dong.append({"kịch bản": ten, "PSI của x₁": psi(X_nen[:, 0], X[:, 0]), "PSI của điểm số": psi(diem_nen, diem),
                 "độ chính xác (%)": 100 * mo_hinh.score(X, y)})
bang_troi = pd.DataFrame(dong).set_index("kịch bản").round(3)
print("hai cột PSI không cần nhãn; cột độ chính xác cần nhãn thật")
bang_troi

Đọc bảng theo cột. Dịch chuyển hiệp biến hiện rõ ở đầu vào. Dịch chuyển nhãn cũng làm phân phối đầu
vào và điểm số dịch đi, vì tỉ lệ hai cụm thay đổi, nên chương nói có thể bắt đầu phát hiện nó mà chưa
cần nhãn, bằng cách theo dõi phân phối dự đoán. Trôi khái niệm thì để nguyên $p(x)$, nên cả hai phép
theo dõi không cần nhãn chỉ thấy nhiễu lấy mẫu, trong khi độ chính xác sụt. Chương nói đúng như vậy:
trôi khái niệm cần nhãn chuẩn (ground truth) để phát hiện. Đây là thất bại thứ ba trong ba thất bại của chương, hệ
thống phát hiện gian lận bỏ sót một kiểu tấn công mới.

## 6. Nhất quán giữa huấn luyện và phục vụ

Chương nêu yêu cầu bắt buộc về tính nhất quán: các tham số tính trên dữ liệu huấn luyện, như hằng số
chuẩn hoá, từ điển mã hoá, bảng từ vựng, phải được lưu lại và dùng lại khi phục vụ. Ngộ nhận phổ biến
là chỉ cần dùng chung mã; thật ra đây là bài toán đồng bộ trạng thái. Vi phạm nó sinh ra lệch giữa huấn
luyện và phục vụ (training-serving skew).

Sổ tay chuẩn hoá logarit của số tiền giao dịch về điểm z, và một quy tắc **giả định** gắn cờ giao dịch
có điểm z trên 2.5 để rà soát. Khi phục vụ có hai cách: dùng trung bình và độ lệch chuẩn đã lưu từ huấn
luyện, hoặc tính lại chúng trên lô đang phục vụ. Hai lô **giả định** 500 giao dịch: một lô ban đêm, số
tiền nhỏ hơn ban ngày, và một lô có một đợt gian lận, 80 giao dịch lớn bất thường.

**Dự đoán:** với lô có đợt gian lận, cách tính lại trên lô gắn cờ được bao nhiêu giao dịch lớn?

In [ ]:
rng = np.random.default_rng(12)
log_hl = rng.normal(3.5, 1.0, 50_000)                        # log số tiền khi huấn luyện
TRUNG_BINH, DO_LECH = log_hl.mean(), log_hl.std()            # trạng thái phải được lưu lại
NGUONG_Z = 2.5                                               # giả định
LO = {"lô ban đêm": rng.normal(3.0, 0.8, 500),
      "lô có đợt gian lận": np.concatenate([rng.normal(3.5, 1.0, 420), rng.normal(7.0, 0.25, 80)])}


def z_luu(x):
    return (x - TRUNG_BINH) / DO_LECH


def z_tinh_lai(x):
    return (x - x.mean()) / x.std()


bien_z = np.linspace(-4, 4, 33)
for ten, lo in LO.items():
    co_dung, co_sai = z_luu(lo) > NGUONG_Z, z_tinh_lai(lo) > NGUONG_Z
    # phân kỳ KL giữa phân phối đặc trưng do hai phép biến đổi sinh ra, theo định nghĩa 1.3
    d = kl(ty_le_o(z_tinh_lai(lo), bien_z), ty_le_o(z_luu(lo), bien_z))
    print(f"{ten}: tham số đã lưu gắn cờ {co_dung.sum()}, tính lại trên lô gắn cờ {co_sai.sum()}; "
          f"{(co_dung != co_sai).sum()} giao dịch nhận quyết định khác nhau; D_KL(g_serve ‖ g_train) = {d:.2f} nats")
lon = LO["lô có đợt gian lận"][420:]
print(f"80 giao dịch lớn: điểm z theo tham số đã lưu từ {z_luu(lon).min():.1f}, "
      f"theo lô chỉ tới {z_tinh_lai(LO['lô có đợt gian lận'])[420:].max():.1f}")
assert (z_luu(lon) > NGUONG_Z).all()
with np.errstate(invalid="ignore", divide="ignore"):
    mot_yeu_cau = z_tinh_lai(LO["lô ban đêm"][:1])
print(f"một yêu cầu đơn lẻ, tính lại trên chính nó: điểm z = {mot_yeu_cau[0]} (độ lệch chuẩn của một số là 0)")
assert np.isnan(mot_yeu_cau[0])

Tính lại trên lô làm mỗi lô tự trở thành "bình thường". Lô ban đêm toàn số tiền nhỏ được kéo giãn cho có
trung bình 0 và độ lệch chuẩn 1, nên vài giao dịch không có gì lạ so với lịch sử bị đẩy lên trên ngưỡng.
Lô có đợt gian lận thì ngược lại: chính các giao dịch lớn kéo trung bình và độ lệch chuẩn của lô lên,
nên chúng tự che mình dưới ngưỡng. Một yêu cầu đơn lẻ thì không có độ lệch chuẩn để chia. Cả ba lỗi
biến mất khi trạng thái huấn luyện được lưu và dùng lại.

Hai tính chất nữa của phép biến đổi: biến đổi lặp lại không đổi kết quả (idempotent transformation),
tức chạy hai lần cho cùng trạng thái như chạy một lần, và tất định, tức cùng đầu vào cùng trạng thái
đã ghi lại thì cùng đầu ra. Chương lấy ví dụ một công việc hỏng giữa chừng rồi chạy lại: nối thêm
(append) thì nhân đôi bản ghi, còn chèn hoặc cập nhật (upsert) thì cho đúng trạng thái cuối.

In [ ]:
ban_ghi = [(f"giao_dich_{i:04d}", float(i) * 1.5) for i in range(1000)]


def chay_cong_viec(dich, ban_ghi, kieu, hong_o=None):
    for i, (khoa, gia_tri) in enumerate(ban_ghi):
        if i == hong_o:
            raise RuntimeError("mất kết nối giữa chừng")
        if kieu == "nối thêm":
            dich.append((khoa, gia_tri))
        else:
            dich[khoa] = gia_tri


for kieu, dich in (("nối thêm", []), ("chèn hoặc cập nhật", {})):
    try:
        chay_cong_viec(dich, ban_ghi, kieu, hong_o=600)
    except RuntimeError:
        chay_cong_viec(dich, ban_ghi, kieu)                  # chạy lại từ đầu
    print(f"{kieu}: {len(dich):,} bản ghi sau một lần hỏng và một lần chạy lại (đúng là 1,000)")
    assert len(dich) == (1600 if kieu == "nối thêm" else 1000)


def bam(x):
    return hashlib.sha256(np.ascontiguousarray(x).tobytes()).hexdigest()[:12]


ngay_dang_ky = np.array([date(2026, 1, 1).toordinal() + int(k) for k in np.random.default_rng(13).integers(0, 200, 5)])


def so_ngay_tu_dang_ky(hom_nay):
    return hom_nay.toordinal() - ngay_dang_ky


print("đặc trưng tính theo đồng hồ của lần chạy:",
      bam(so_ngay_tu_dang_ky(date(2026, 10, 4))), "rồi", bam(so_ngay_tu_dang_ky(date(2026, 10, 5))))
NGAY_CHOT = date(2026, 9, 30)                                # ghi vào cấu hình, có phiên bản
print("đặc trưng tính theo ngày chốt đã ghi lại:",
      bam(so_ngay_tu_dang_ky(NGAY_CHOT)), "rồi", bam(so_ngay_tu_dang_ky(NGAY_CHOT)))
assert bam(so_ngay_tu_dang_ky(NGAY_CHOT)) == bam(so_ngay_tu_dang_ky(NGAY_CHOT))

Đọc đồng hồ hệ thống bên trong một phép biến đổi làm nó mất tính tất định: chạy lại hôm sau ra một tập
đặc trưng khác, không ai đổi dòng mã nào. Ghi ngày chốt vào cấu hình có phiên bản, như chương nói về
tham số của đặc trưng âm thanh, thì lần chạy nào cũng ra cùng một mã băm.

## 7. Nhãn sai và giá trị thiếu

Bảng 10 của chương coi tỉ lệ nhãn sai đã biết trên 1 % là cảnh báo và trên 5 % là nghiêm trọng, và lấy
ví dụ một tập có 3 % nhãn sai đã biết là nợ chất lượng (quality debt). Sổ tay đo xem nhãn sai làm mô hình kém đi bao
nhiêu, với hồi quy logistic trên `load_digits`, đánh giá trên tập kiểm tra có nhãn sạch. Hai kiểu nhãn
sai:

* **ngẫu nhiên:** nhãn bị thay bằng một chữ số khác bất kỳ;
* **có hệ thống:** nhãn bị đổi theo những cặp dễ nhầm (3 thành 5, 4 thành 9, 7 thành 1, và ngược lại),
  như khi một người gán nhãn hiểu sai hướng dẫn.

**Dự đoán:** cùng một tỉ lệ nhãn sai, kiểu nào hại hơn?

In [ ]:
NHAN_CANH_BAO = sach(1, trich="Known label error rate | > 1% | > 5%")
NHAN_NGHIEM_TRONG = sach(5, trich="Known label error rate | > 1% | > 5%")
NHAN_NO = sach(3, trich="3 percent known label errors")
Xh, Xk, yh, yk = train_test_split(X_so / 16, y_so, test_size=0.3, stratify=y_so, random_state=0)
CAP = {3: 5, 5: 3, 4: 9, 9: 4, 7: 1, 1: 7}


def lam_nhieu(y, ty_le, kieu, rng):
    y = y.copy()
    du_dieu_kien = np.flatnonzero(np.isin(y, list(CAP))) if kieu == "có hệ thống" else np.arange(len(y))
    chon = rng.choice(du_dieu_kien, min(len(du_dieu_kien), int(round(ty_le * len(y)))), replace=False)
    if kieu == "có hệ thống":
        y[chon] = [CAP[v] for v in y[chon]]
    else:
        y[chon] = (y[chon] + rng.integers(1, 10, len(chon))) % 10
    return y


def do_chinh_xac(Xh, yh, Xk=Xk):
    return 100 * LogisticRegression(max_iter=3000).fit(Xh, yh).score(Xk, yk)


CAC_TY_LE = [0, 0.01, 0.03, 0.05, 0.1, 0.2, 0.3]
KQ_NHAN = {}
for kieu in ("ngẫu nhiên", "có hệ thống"):
    rng = np.random.default_rng(14)
    KQ_NHAN[kieu] = [np.mean([do_chinh_xac(Xh, lam_nhieu(yh, p, kieu, rng)) for _ in range(4)]) for p in CAC_TY_LE]
    print(f"{kieu:>12}: " + ", ".join(f"{100 * p:.0f} % → {a:.1f} %" for p, a in zip(CAC_TY_LE, KQ_NHAN[kieu])))

Giá trị thiếu là kiểu hỏng thứ hai. Sổ tay xoá ngẫu nhiên một phần điểm ảnh ở cả tập huấn luyện lẫn tập
kiểm tra, rồi thử hai cách xử lý: điền bằng trung bình của cột tính trên tập huấn luyện (một trạng thái
phải lưu lại, như ở phần 6), hoặc bỏ mọi dòng có ô thiếu. Cách thứ hai nghe an toàn, nhưng với 64 cột
mỗi dòng, xác suất một dòng còn nguyên là $(1 - p)^{64}$.

In [ ]:
CAC_THIEU = [0, 0.02, 0.05, 0.1, 0.2, 0.4]
rng = np.random.default_rng(15)
KQ_THIEU = []
for p in CAC_THIEU:
    Xh_t = np.where(rng.random(Xh.shape) < p, np.nan, Xh)
    Xk_t = np.where(rng.random(Xk.shape) < p, np.nan, Xk)
    dien = SimpleImputer(strategy="mean", keep_empty_features=True).fit(Xh_t)
    con_lai = ~np.isnan(Xh_t).any(axis=1)
    KQ_THIEU.append({"thiếu (%)": 100 * p, "điền trung bình: độ chính xác (%)": do_chinh_xac(dien.transform(Xh_t), yh, dien.transform(Xk_t)),
                     "bỏ dòng: số dòng huấn luyện còn lại": int(con_lai.sum()),
                     "(1 − p)^64 dự đoán": round(len(Xh) * (1 - p) ** 64)})
bang_thieu = pd.DataFrame(KQ_THIEU).round(1)
bang_thieu

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.5, 3.9))
for (kieu, kq), mau in zip(KQ_NHAN.items(), MAU):
    ax1.plot(100 * np.array(CAC_TY_LE), kq, "o-", color=mau, lw=2, ms=5, label=f"nhãn sai {kieu}")
for x in (NHAN_CANH_BAO, NHAN_NO, NHAN_NGHIEM_TRONG):
    ax1.axvline(x, color="0.6", lw=0.8, ls=":")
ax1.text(10.5, 67, "đường chấm: 1 % cảnh báo, 5 % nghiêm trọng\n(bảng 10), 3 % ví dụ nợ chất lượng", fontsize=7, color="0.3")
ax1.set_xlabel("tỉ lệ nhãn sai trong tập huấn luyện (%)")
ax1.set_ylabel("độ chính xác trên tập kiểm tra sạch (%)")
ax1.set_title("Nhãn sai", fontsize=10)
ax1.legend(loc="center left", fontsize=8, frameon=False)
ax2.plot(bang_thieu["thiếu (%)"], bang_thieu["điền trung bình: độ chính xác (%)"], "o-", color=MAU[0], lw=2, ms=5)
ax2.set_xlabel("tỉ lệ ô bị thiếu (%)")
ax2.set_ylabel("độ chính xác, điền trung bình (%)")
con = dict(zip(bang_thieu["thiếu (%)"], bang_thieu["bỏ dòng: số dòng huấn luyện còn lại"]))
ax2.text(0.04, 0.06, "nếu bỏ mọi dòng có ô thiếu, tập huấn luyện\n"
         f"{con[0.0]:,} dòng còn {con[2.0]:,} ở 2 %, {con[5.0]:,} ở 5 %, {con[10.0]:,} ở 10 %",
         transform=ax2.transAxes, fontsize=7.5, color="0.25")
ax2.set_title("Giá trị thiếu", fontsize=10)
fig.suptitle("Hồi quy logistic trên load_digits (tính toán, có hạt giống)", fontsize=11)
plt.tight_layout()
plt.show()

Ở tỉ lệ nhỏ, hai đường gần nhau và lẫn trong nhiễu của việc chọn ngẫu nhiên nhãn nào bị sai. Ở tỉ lệ
lớn, chúng tách hẳn, vì hai kiểu sai dạy mô hình hai điều khác nhau. Nhãn sai ngẫu nhiên rải đều
trên mọi lớp, nên phần lớn bị các nhãn đúng xung quanh lấn át. Nhãn sai có hệ thống luôn đẩy về cùng
một hướng, nên chúng dạy mô hình một quy tắc sai nhất quán. Đó cũng là lý do chương lo về vòng phản
hồi của nợ chất lượng: dự đoán của một mô hình học từ nhãn sai trở thành dữ liệu huấn luyện của hệ
thống phía sau. Với giá trị thiếu, bảng cho thấy cách "bỏ dòng có ô thiếu" mất gần hết tập huấn luyện
ngay ở tỉ lệ thiếu nhỏ, đúng như $(1-p)^{64}$ dự đoán.

## 8. Đo nợ dữ liệu

Chương mở phần sức khoẻ dữ liệu bằng một hệ thống phát hiện từ khoá ra mắt với 98 % độ chính xác, sáu
tháng sau còn 94 %, mà không ai đổi mô hình. Để biến nợ dữ liệu thành thứ theo dõi được như một mục
tiêu mức dịch vụ (service-level objective, SLO), bảng 10 cho các ngưỡng minh hoạ, như tỉ lệ tập dữ liệu
có thẻ dữ liệu (data card). Sổ tay chấm một danh mục **giả định** gồm bốn tập dữ
liệu theo bảng đó.

In [ ]:
DO_CHINH_XAC_KWS = (sach(98, nguon="98 % → 94 % sau sáu tháng"), sach(94, nguon="98 % → 94 % sau sáu tháng"))
print(f"KWS: {DO_CHINH_XAC_KWS[0]} % → {DO_CHINH_XAC_KWS[1]} % sau sáu tháng, "
      f"tức tỉ lệ lỗi tăng {(100 - DO_CHINH_XAC_KWS[1]) / (100 - DO_CHINH_XAC_KWS[0]):.0f} lần")

# chỉ số: (cảnh báo, nghiêm trọng, chiều xấu). "<" nghĩa là thấp hơn là xấu.
BANG_10 = {
    "% tập có thẻ dữ liệu": (sach(80, trich="% datasets with data cards | < 80% | < 50%"),
                             sach(50, trich="% datasets with data cards | < 80% | < 50%"), "<"),
    "% cột có mô tả": (sach(90, trich="% columns with descriptions | < 90% | < 70%"),
                       sach(70, trich="% columns with descriptions | < 90% | < 70%"), "<"),
    "số nhánh phiên bản giản đồ": (sach(3, trich="Schema version branches | > 3 | > 10"),
                                   sach(10, trich="Schema version branches | > 3 | > 10"), ">"),
    "% phép biến đổi bọc try/catch": (sach(20, trich="% transformations with try/catch | > 20% | > 50%"),
                                      sach(50, trich="% transformations with try/catch | > 20% | > 50%"), ">"),
    "% nhãn sai đã biết": (NHAN_CANH_BAO, NHAN_NGHIEM_TRONG, ">"),
    "số ngày từ lần huấn luyện lại": (sach(90, trich="Days since last retraining | > 90 days | > 365 days"),
                                      sach(365, trich="Days since last retraining | > 90 days | > 365 days"), ">"),
    "PSI so với dữ liệu huấn luyện": (PSI_CANH_BAO, PSI_NGHIEM_TRONG, ">"),
}
DANH_MUC = pd.DataFrame({          # giả định
    "giao dịch thẻ": [100, 95, 2, 10, 0.5, 30, 0.04],
    "âm thanh KWS": [100, 85, 4, 25, 3.0, 120, 0.18],
    "nhật ký nhấp chuột": [60, 92, 12, 35, 0.8, 45, 0.31],
    "ảnh y tế": [40, 65, 1, 55, 6.0, 400, 0.09],
}, index=list(BANG_10))


def muc_no(chi_so, gia_tri):
    canh_bao, nghiem_trong, chieu = BANG_10[chi_so]
    xau = (lambda a, b: a < b) if chieu == "<" else (lambda a, b: a > b)
    return "nghiêm trọng" if xau(gia_tri, nghiem_trong) else "cảnh báo" if xau(gia_tri, canh_bao) else "ổn"


cham = DANH_MUC.apply(lambda cot: pd.Series([f"{v:g} · {muc_no(k, v)}" for k, v in cot.items()], index=cot.index))
cham

Bảng chấm biến một cảm giác mơ hồ ("dữ liệu này có vẻ cũ") thành một hạng mục có ngưỡng và có người chịu
trách nhiệm. Chương nhấn mạnh các ngưỡng là ví dụ cần hiệu chỉnh theo rủi ro của từng hệ thống, trừ dòng
PSI dùng các dải theo dõi trôi thường gặp.

Chương còn đưa phương trình 8 để thấy độ nhạy của nợ với một tốc độ tích luỹ đều $r$ mỗi kỳ:
$\text{Debt}_n \approx \text{Debt}_0 (1 + r)^n$. Chương nói rõ đây là một mô hình lập kế hoạch, không phải
một quy luật thực nghiệm. Sổ tay quét $r$ (các giá trị là **giả định**) qua 8 quý.

In [ ]:
quy = np.arange(0, 9)
fig, ax = plt.subplots(figsize=(8, 3.6))
for r, mau in zip((0.05, 0.10, 0.20), MAU):
    ax.plot(quy, (1 + r) ** quy, "o-", color=mau, lw=2, ms=5, label=f"r = {r:.0%} mỗi quý (giả định)")
    ax.text(8.15, (1 + r) ** 8, f"×{(1 + r) ** 8:.2f}", fontsize=8, va="center")
ax.set_xlim(-0.3, 9)
ax.set_xlabel("số quý không trả nợ")
ax.set_ylabel("nợ sau n quý / nợ ban đầu")
ax.set_title("Phương trình 8: một mô hình lập kế hoạch, không phải quy luật")
ax.legend(loc="upper left", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()
for r in (0.05, 0.10, 0.20):
    print(f"r = {r:.0%}: nợ gấp đôi sau {math.log(2) / math.log(1 + r):.1f} quý")

## Thử thêm

1. Ở phần 2, thay `KNeighborsClassifier(n_neighbors=1)` bằng `LogisticRegression(max_iter=3000)`. Một mô
   hình ít ghi nhớ hơn có bị rò rỉ đánh lừa ít hơn không?
2. Ở phần 4, đổi `so_o` của PSI từ 10 thành 50 rồi chạy lại vòng quét $\delta$. PSI của cùng một dịch
   chuyển có đổi không? Điều đó nói gì về việc dùng chung một ngưỡng PSI cho mọi đặc trưng?
3. Ở phần 5, giảm `goc_xoay` xuống 20 độ. Độ chính xác còn sụt bao nhiêu, và bạn cần bao nhiêu nhãn để
   thấy sự sụt đó?

## Tóm lại

* Một khế ước dữ liệu ở ranh giới chặn được lỗi cơ học như mã bưu chính mất số 0 hay tuổi 150, trước khi
  chúng thành thác lỗi dữ liệu.
* Rò rỉ qua lằn ranh giữa các tập, bằng biến thể tăng cường hay bằng đặc trưng tính sau sự kiện, làm tỉ lệ
  lỗi đo được thấp hơn thật; chia theo bản ghi gốc và truy xuất theo thời điểm loại bỏ nó.
* K-S và PSI bắt được lỗi ngữ nghĩa mà kiểm tra từng bản ghi để lọt, nhưng độ nhạy của chúng tăng theo cỡ
  mẫu, nên ngưỡng phải hiệu chỉnh theo hệ thống; trôi khái niệm chỉ lộ ra khi có nhãn.
* Trạng thái huấn luyện phải được lưu và dùng lại khi phục vụ, và phép biến đổi phải chạy lại không đổi kết
  quả và không đọc đồng hồ.
* Khi tỉ lệ nhãn sai tăng, nhãn sai có hệ thống hại hơn hẳn nhãn sai ngẫu nhiên cùng tỉ lệ; nợ dữ liệu
  theo dõi được khi mỗi loại có một chỉ số và hai ngưỡng.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Kỹ thuật dữ liệu](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch04-data-engineering/), dựng từ chương
[*Data Engineering*](https://mlsysbook.ai/vol1/data_engineering/data_engineering.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_04_data_engr.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.